# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 5: Fine-tuning a Frontier Model

Now we will use OpenAI's API to fine-tune our own private variant of GPT-4.1-nano

In [4]:
# imports

import os

# Set SSL environment variables BEFORE importing httpx/sayari
cert_bundle = os.path.join(os.getcwd(), 'combined-ca-bundle.pem')
os.environ['SSL_CERT_FILE'] = cert_bundle
os.environ['REQUESTS_CA_BUNDLE'] = cert_bundle
os.environ['CURL_CA_BUNDLE'] = cert_bundle

# Hugging Face specific SSL configuration
os.environ['HF_CACERT'] = cert_bundle

import re
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items  import Item
from pricer.evaluator import evaluate
import httpx

http_client = httpx.Client(verify=cert_bundle)


In [6]:
# environment

LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

TypeError: login() got an unexpected keyword argument 'httpx_client'

In [4]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


In [5]:
openai = OpenAI()

# Data size

OpenAI recommends fine-tuning with a small population of 50-100 examples

I'm going to go with 20,000 points.

This cost me $3.42 - you should stick with 100 examples and the cost will be minimal!

In [6]:
# OpenAI recommends fine-tuning with populations of 50-100 examples
# But as our examples are very small, I'm suggesting we go with 100 examples (and 1 epoch)


fine_tune_train = train[:100]
fine_tune_validation = val[:50]

In [7]:
len(fine_tune_train)

100

# Step 1

Prepare our data for fine-tuning in JSONL (JSON Lines) format and upload to OpenAI

In [8]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

In [9]:
messages_for(fine_tune_train[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.'},
 {'role': 'assistant', 'content': '$64.30'}]

In [11]:
# Convert the items into a list of json objects - a "jsonl" string
# Each row represents a message in the form:
# {"messages" : [{"role": "system", "content": "You estimate prices...


def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [12]:
print(make_jsonl(train[:3]))

{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single\u2011piece oil\u2011rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4\" minimum center\u2011to\u2011center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation."}, {"role": "assistant", "content": "$64.30"}]}
{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Mini Electric Air Duster Fan  \nCategory: Electronics  \nBrand: Kica  \nDescription: Ultra\u2011compact 86,000\u202fRPM electric air duster with 11\u202fm/s wind speed for precise cleaning and inflation.  \nDetails: Powered by a 9.99\u202fWh motor, adjustable in four speed levels, it uses three 

In [13]:
# Convert the items into jsonl and write them to a file

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [14]:
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")

In [15]:
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

In [16]:
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

In [17]:
train_file

FileObject(id='file-NVDtnM6Up8aaiN5ua3wjhy', bytes=55120, created_at=1771591301, filename='fine_tune_train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [18]:
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [19]:
validation_file

FileObject(id='file-7qHqkHej8puorcVz4oUHDp', bytes=27637, created_at=1771591342, filename='fine_tune_validation.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

https://platform.openai.com/storage/files/

# Step 2

## And now time to Fine-tune!

In [20]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="pricer"
)

FineTuningJob(id='ftjob-dhAbb29soDDYbwOytbSBceXh', created_at=1771591537, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-JrAdAU6IS68DIE9ne1VFYbQU', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-NVDtnM6Up8aaiN5ua3wjhy', validation_file='file-7qHqkHej8puorcVz4oUHDp', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None)

In [21]:
openai.fine_tuning.jobs.list(limit=1)

SyncCursorPage[FineTuningJob](data=[FineTuningJob(id='ftjob-dhAbb29soDDYbwOytbSBceXh', created_at=1771591537, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-JrAdAU6IS68DIE9ne1VFYbQU', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-NVDtnM6Up8aaiN5ua3wjhy', validation_file='file-7qHqkHej8puorcVz4oUHDp', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None)], has_more=False, object='list')

In [22]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id

In [23]:
job_id

'ftjob-dhAbb29soDDYbwOytbSBceXh'

In [24]:
openai.fine_tuning.jobs.retrieve(job_id)

FineTuningJob(id='ftjob-dhAbb29soDDYbwOytbSBceXh', created_at=1771591537, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-JrAdAU6IS68DIE9ne1VFYbQU', result_files=[], seed=42, status='running', trained_tokens=None, training_file='file-NVDtnM6Up8aaiN5ua3wjhy', validation_file='file-7qHqkHej8puorcVz4oUHDp', estimated_finish=1771592086, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None)

In [36]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

[FineTuningJobEvent(id='ftevent-Nx8OUwtW5ssytGFtXvBJUqBr', created_at=1771592829, level='info', message='The job has successfully completed', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-rSsQCbd9cfmh5yhQZQiiaWdq', created_at=1771592828, level='info', message='Usage policy evaluations completed, model is now enabled for sampling', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-Dm3aDbcQ9grSznKYVqnJlOrK', created_at=1771592828, level='info', message='Moderation checks for snapshot ft:gpt-4.1-nano-2025-04-14:personal:pricer:DBKK3eVu passed.', object='fine_tuning.job.event', data={'blocked': False, 'results': [{'flagged': False, 'category': 'harassment/threatening', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'sexual', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'sexual/minors', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'propaganda', 'enforcement': 'blocking

https://platform.openai.com/finetune


# Step 3

Test our fine tuned model

In [37]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [38]:
fine_tuned_model_name

'ft:gpt-4.1-nano-2025-04-14:personal:pricer:DBKK3eVu'

In [34]:
# The prompt

def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
    ]

In [35]:
# Try this out

test_messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [39]:
# The inference function


def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content

In [40]:
print(test[0].price)
print(gpt_4__1_nano_fine_tuned(test[0]))

219.0
$209.00


In [41]:
evaluate(gpt_4__1_nano_fine_tuned, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$45 $44 $10 $25 $25 $5 $25 $102 $25 $27 $225 $150 $20 $12 $26 $5 $3 $4 $63 $73 $64 $63 $121 $34 $194 $254 $114 $5 $39 $68 $2 $124 $31 $20 $84 $140 $110 $17 $4 $31 $181 $6 $13 $176 $70 $4 $42 $11 $98 $107 $6 $104 $120 $5 $87 $16 $11 $60 $67 $10 $54 $17 $3 $169 $57 $1 $20 $176 $630 $76 $16 $4 $184 $1 $120 $22 $55 $10 $11 $8 $63 $5 $1 $54 $12 $10 $12 $62 $46 $66 $13 $211 $1 $18 $3 $121 $0 $619 $2 $209 $30 $110 $40 $25 $116 $153 $17 $385 $11 $100 $21 $251 $58 $78 $124 $13 $21 $20 $2539 $48 $14 $185 $80 $5 $34 $90 $0 $17 $30 $64 $63 $82 $83 $0 $318 $10 $52 $3 $134 $38 $9 $81 $27 $6 $130 $75 $23 $718 $91 $8 $8 $64 $23 $93 $116 $121 $126 $34 $18 $29 $410 $10 $33 $4 $491 $23 $128 $33 $20 $23 $9 $2 $270 $5 $44 $91 $3 $42 $324 $17 $3 $20 $251 $2 $84 $1 $94 $22 $35 $7 $5 $58 $20 $61 $66 $153 $9 $52 $39 $9 

In [ ]:
# 96.58 - mini 200
# 79.29 - mini 2000
# 82.26 - nano 2000
# 67.75 - nano 20,000